## Single Responsibility Principle (SRP) & API Chaining

This notebook explores three critical concepts in modern software development:
1. **Single Responsibility Principle (SRP)** - Each function/module should do ONE thing well
2. **API Chaining** - Sequential API calls where each step depends on the previous
3. **Function Anatomy** - Designing functions according to APCSP Procedural Abstraction requirements

### Why This Matters

- **Maintainability**: Code that follows SRP is easier to debug and modify
- **Testability**: Functions with single responsibilities are easier to test
- **Reusability**: Well-designed functions can be reused across different contexts
- **Collaboration**: Clear function purposes make team development smoother

## Part 1: Single Responsibility Principle (SRP)

### What is SRP?

The Single Responsibility Principle states that **a function or class should have one, and only one, reason to change**. In practice, this means each function should do exactly ONE thing.

### ❌ Bad Example: Multiple Responsibilities

```js
// This function does TOO MANY things!
function processUserData(userId) {
    // Responsibility 1: Fetch data from API
    const response = fetch(`/api/user/${userId}`);
    const userData = response.json();
    
    // Responsibility 2: Validate data
    if (!userData.email || !userData.name) {
        throw new Error('Invalid data');
    }
    
    // Responsibility 3: Transform data
    const displayName = userData.name.toUpperCase();
    
    // Responsibility 4: Update UI
    document.getElementById('userName').textContent = displayName;
    
    // Responsibility 5: Log analytics
    console.log('User viewed:', userId);
    
    return userData;
}
```

**Problems:**
- Hard to test (mixes API calls, validation, UI updates)
- Can't reuse validation logic elsewhere
- Changes to UI require modifying a function that also handles API calls
- Violates separation of concerns

### ✅ Good Example: Single Responsibilities

Breaking the previous function into focused, single-purpose functions:

```js
// Responsibility 1: Fetch data from API
async function fetchUserById(userId) {
    const response = await fetch(`/api/user/${userId}`);
    if (!response.ok) throw new Error(`Failed to fetch user ${userId}`);
    return response.json();
}

// Responsibility 2: Validate user data
function validateUserData(userData) {
    if (!userData.email || !userData.name) {
        throw new Error('User data missing required fields');
    }
    return true;
}

// Responsibility 3: Transform data for display
function formatDisplayName(name) {
    return name.toUpperCase();
}

// Responsibility 4: Update UI element
function updateUserNameDisplay(displayName) {
    const element = document.getElementById('userName');
    if (element) element.textContent = displayName;
}

// Responsibility 5: Log analytics event
function logUserView(userId) {
    console.log('User viewed:', userId);
}

// Sequence Orchestrator function (coordinates the workflow)
async function displayUserProfile(userId) {
    const userData = await fetchUserById(userId);
    validateUserData(userData);
    const displayName = formatDisplayName(userData.name);
    updateUserNameDisplay(displayName);
    logUserView(userId);
    return userData;
}
```

**Benefits:**
- ✅ Each function has ONE clear purpose
- ✅ Easy to test individually (`formatDisplayName` doesn't need API or DOM)
- ✅ Reusable (can use `validateUserData` in other contexts)
- ✅ Easy to modify (change UI without touching API logic)
- ❌ **Poor error handling** - What if `fetchUserById` fails? What if `validateUserData` throws? The UI will break with no feedback to the user!

**This leads us to API Chaining...** where we can add centralized error handling with `.catch()`!


## Part 2: API Chaining Pattern

### What is API Chaining?

API chaining is a pattern where **each API call depends on data from the previous call**. This creates a sequential workflow where Step 2 cannot execute until Step 1 completes.

### Real-World Example: User → Groups

**Scenario:** To display a user's groups, we need:
1. **Step 1:** Fetch user identity (to get `personId`)
2. **Step 2:** Use that `personId` to fetch groups
3. **Step 3:** Render the groups in the UI

**Why Chaining?** The groups API endpoint requires `personId`, which we don't have until Step 1 completes.

### The Chain Structure

```
Step 1: GET /api/person/get
   ↓
   Returns: { id: 123, uid: "student1", name: "Alice" }
   ↓
Step 2: GET /api/groups/person/123  (using id from Step 1)
   ↓
   Returns: [{ id: 5, name: "CS Team", members: [...] }]
   ↓
Step 3: Render groups to DOM
```

## Annotated API Chaining Code

Below is the complete chained pattern with detailed annotations explaining each step:

```javascript
/**
 * CHAINED API PATTERN: User Identity → User Groups → UI Rendering
 * 
 * Pattern: fetch().then().then().then().catch()
 * Each .then() represents a sequential step that depends on the previous
 */

// STEP 1: Fetch User Identity
fetch(`${javaURI}/api/person/get`, fetchOptions)
    .then(response => {
        // SRP Focus: Handle authentication/response validation
        
        // Authentication check - stop chain if not logged in
        if (response.status === 401) {
            springGroupInfo.innerHTML = `<p class="text-gray-400">Please sign in.</p>`;
            groupSearchInput.disabled = true;
            return null; // STOP THE CHAIN - no point continuing
        }
        
        // Error handling
        if (!response.ok) {
            throw new Error(`Failed to fetch user: ${response.status}`);
        }
        
        // Parse JSON and pass to next step
        return response.json();
    })
    
    // STEP 2: Extract User Data & Request Groups
    .then(personData => {
        // SRP Focus: Validate user data and prepare next request
        
        if (!personData) return null; // Chain was stopped in Step 1
        
        // Cache user identity for later use (join/leave operations)
        currentPersonId = personData.id;
        currentUid = personData.uid;

        // Data validation
        if (!currentPersonId || !currentUid) {
            springGroupInfo.innerHTML = `<p>Unable to verify identity.</p>`;
            return null; // STOP THE CHAIN
        }

        // CHAIN DEPENDENCY: This request REQUIRES currentPersonId from Step 1
        // We couldn't make this call until we had the personId
        return fetch(`${javaURI}/api/groups/person/${currentPersonId}`, fetchOptions);
    })
    
    // STEP 3: Handle Group Response (with fallback)
    .then(groupResp => {
        // SRP Focus: Handle group response and fallback logic
        
        if (!groupResp) return null; // Chain stopped earlier
        
        // Success path
        if (groupResp.ok) {
            return groupResp.json();
        } 
        
        // Fallback: If endpoint unavailable, fetch all groups and filter client-side
        else if (groupResp.status === 404) {
            console.warn('Using fallback: fetching all groups');
            return fetch(`${javaURI}/api/groups`, fetchOptions)
                .then(fallbackResp => {
                    if (!fallbackResp.ok) return [];
                    return fallbackResp.json().then(allGroups => {
                        // Client-side filtering (less efficient but works)
                        return (Array.isArray(allGroups) ? allGroups : [])
                            .filter(group =>
                                Array.isArray(group.members) && 
                                group.members.some(m => m.uid === currentUid)
                            );
                    });
                });
        }
        
        throw new Error('Failed to fetch group data');
    })
    
    // STEP 4: Render Groups to UI
    .then(userGroups => {
        // SRP Focus: Transform data and update DOM
        
        if (!userGroups) return; // Chain stopped
        
        const groups = Array.isArray(userGroups) ? userGroups : [];

        // Build lookup set for search feature
        groups.forEach(group => userGroupIds.add(group.id));

        // Render groups or empty state
        if (groups.length > 0) {
            springGroupInfo.innerHTML = groups.map(group => `
                <div>
                    <label class="block text-sm font-medium">Group Name</label>
                    <input type="text" readonly value="${group.name}"
                        class="w-full px-4 py-2 rounded-lg bg-neutral-700">
                </div>
                <div>
                    <label class="block text-sm font-medium">Period</label>
                    <input type="text" readonly value="${group.period}"
                        class="w-full px-4 py-2 rounded-lg bg-neutral-700">
                </div>
                <div class="col-span-2">
                    <label class="block text-sm font-medium">Members</label>
                    <div class="w-full px-4 py-2 rounded-lg bg-neutral-700">
                        ${group.members.map(m => `${m.name} (${m.uid})`).join(', ')}
                    </div>
                </div>
                <div class="col-span-2">
                    <button type="button" 
                        onclick="leaveGroup('${group.id}', ${currentPersonId})"
                        class="px-4 py-2 bg-red-400 rounded-lg hover:bg-red-600">
                        Leave Group
                    </button>
                </div>
            `).join('');
        } else {
            springGroupInfo.innerHTML = `<p class="text-gray-400">No groups found.</p>`;
        }

        // Enable search (uses cached currentPersonId)
        setupGroupSearch(groupSearchInput, groupResultsList, 
            joinGroupStatus, userGroupIds, currentPersonId);
    })
    
    // CATCH: Handle ANY error in the entire chain
    .catch(error => {
        // SRP Focus: Centralized error handling for entire workflow
        console.error("Error in API chain:", error);
        springGroupInfo.innerHTML = `<p class="text-red-400">Error loading groups.</p>`;
        groupSearchInput.disabled = true;
    });
```

### Key Observations

1. **Sequential Dependencies:** Each `.then()` depends on data from previous steps
2. **Early Exits:** `return null` stops the chain when conditions aren't met
3. **Single Responsibilities:** Each `.then()` has ONE focus (validate, fetch, render)
4. **Centralized Error Handling:** One `.catch()` handles all errors in chain
5. **Data Caching:** `currentPersonId` saved for use in other operations



## Part 3: APCSP Function Anatomy (Procedural Abstraction)

### APCSP Requirements for Functions

According to APCSP Performance Task Requirements, a good procedure (function) must demonstrate:

1. **Parameters** - Input values that customize function behavior
2. **Algorithm** - Sequenced instructions implementing the function's purpose
3. **Return Value** - Output that can be used by calling code
4. **Abstraction** - Hides complexity, provides clear interface

### Example: Following SRP + APCSP Requirements

Let's design a function that validates group membership and follows both SRP and APCSP standards.


In [4]:
%%js

/**
 * APCSP Procedural Abstraction Example
 * Function: isUserInGroup
 * 
 * Purpose: Determine if a specific user is a member of a given group
 * 
 * PARAMETERS (inputs that customize behavior):
 * @param {string} uid - The user's unique identifier (GitHub ID)
 * @param {object} group - The group object containing members array
 * 
 * RETURN VALUE (output for calling code):
 * @returns {boolean} - true if user is in group, false otherwise
 * 
 * ALGORITHM (sequenced steps):
 * 1. Validate inputs exist
 * 2. Check if group has members array
 * 3. Search members array for matching uid
 * 4. Return boolean result
 * 
 * ABSTRACTION:
 * - Hides complexity of null checking and array iteration
 * - Provides simple yes/no answer
 * - Reusable across different contexts
 */
function isUserInGroup(uid, group) {
    // Step 1: Input validation (defensive programming)
    if (!uid || !group) {
        return false; // Invalid inputs = not in group
    }
    
    // Step 2: Ensure members array exists
    if (!Array.isArray(group.members)) {
        return false; // No members array = user can't be member
    }
    
    // Step 3: Search for user in members
    // Algorithm: Use .some() to check if ANY member matches uid
    const isMember = group.members.some(member => member.uid === uid);
    
    // Step 4: Return boolean result
    return isMember;
}

// USAGE EXAMPLES:

// Example 1: User is in group
const group1 = {
    id: 5,
    name: "CS Team",
    members: [
        { uid: "student1", name: "Alice" },
        { uid: "student2", name: "Bob" }
    ]
};

console.log(isUserInGroup("student1", group1)); // true
console.log(isUserInGroup("student3", group1)); // false

// Example 2: Invalid inputs (demonstrates abstraction)
console.log(isUserInGroup(null, group1));      // false (handles null gracefully)
console.log(isUserInGroup("student1", null));  // false (handles null gracefully)
console.log(isUserInGroup("student1", {}));    // false (handles missing members)

<IPython.core.display.Javascript object>

### Breaking Down the Function (APCSP Analysis)

#### 1. Parameters (Customization)
```javascript
function isUserInGroup(uid, group)
```
- **uid**: Changes WHO we're searching for
- **group**: Changes WHERE we're searching
- Different combinations of inputs produce different outputs

#### 2. Algorithm (Sequenced Steps)
```javascript
// Step 1: Validate inputs
if (!uid || !group) return false;

// Step 2: Check members array exists  
if (!Array.isArray(group.members)) return false;

// Step 3: Search array for matching uid
const isMember = group.members.some(member => member.uid === uid);

// Step 4: Return result
return isMember;
```

#### 3. Return Value (Output)
- Returns `boolean` (true/false)
- Calling code can use result in conditionals:
  ```javascript
  if (isUserInGroup(currentUid, group)) {
      console.log("Already a member!");
  }
  ```

#### 4. Abstraction (Hiding Complexity)
- **Hides:** Null checking, array validation, iteration logic
- **Provides:** Simple yes/no answer
- **Benefit:** Calling code doesn't need to know HOW the search works

### Why This Follows SRP

**Single Responsibility:** This function does ONE thing - checks membership.

It does NOT:
- ❌ Fetch data from API
- ❌ Update the UI
- ❌ Send analytics events
- ❌ Modify the group

**Result:** Easy to test, reuse, and maintain!

## Part 4: Refactoring the Chain with SRP

Let's refactor the chained API example into smaller SRP-compliant functions:

In [ ]:
/**
 * SRP-Compliant Functions for Group Management
 * Each function has exactly ONE responsibility
 */

// ============================================
// RESPONSIBILITY: Fetch user identity from API
// ============================================
async function fetchCurrentUser(javaURI, fetchOptions) {
    const response = await fetch(`${javaURI}/api/person/get`, fetchOptions);
    
    if (response.status === 401) {
        throw new Error('AUTHENTICATION_REQUIRED');
    }
    
    if (!response.ok) {
        throw new Error(`Failed to fetch user: ${response.status}`);
    }
    
    return response.json();
}

// ============================================
// RESPONSIBILITY: Validate user data structure
// ============================================
function validateUserIdentity(personData) {
    if (!personData || !personData.id || !personData.uid) {
        throw new Error('Invalid user data: missing id or uid');
    }
    return {
        personId: personData.id,
        uid: personData.uid,
        name: personData.name
    };
}

// ============================================
// RESPONSIBILITY: Fetch groups for a specific user
// ============================================
async function fetchUserGroups(personId, javaURI, fetchOptions) {
    const response = await fetch(
        `${javaURI}/api/groups/person/${personId}`, 
        fetchOptions
    );
    
    if (response.ok) {
        return response.json();
    }
    
    if (response.status === 404) {
        // Endpoint not available - return empty array
        console.warn('Groups endpoint not found');
        return [];
    }
    
    throw new Error('Failed to fetch groups');
}

// ============================================
// RESPONSIBILITY: Filter groups by user membership
// ============================================
function filterGroupsByMembership(groups, uid) {
    if (!Array.isArray(groups)) return [];
    
    return groups.filter(group =>
        Array.isArray(group.members) && 
        group.members.some(member => member.uid === uid)
    );
}

// ============================================
// RESPONSIBILITY: Render groups to DOM
// ============================================
function renderGroupsToUI(groups, containerElement, personId) {
    if (!containerElement) return;
    
    if (groups.length === 0) {
        containerElement.innerHTML = `
            <p class="text-gray-400 text-center">No groups found.</p>
        `;
        return;
    }
    
    containerElement.innerHTML = groups.map(group => `
        <div class="group-card">
            <h3>${group.name}</h3>
            <p>Period: ${group.period}</p>
            <p>Members: ${group.members.map(m => m.name).join(', ')}</p>
            <button onclick="leaveGroup('${group.id}', ${personId})">
                Leave Group
            </button>
        </div>
    `).join('');
}

// ============================================
// RESPONSIBILITY: Display error message to user
// ============================================
function showErrorMessage(containerElement, message) {
    if (!containerElement) return;
    containerElement.innerHTML = `
        <p class="text-red-400 text-center">${message}</p>
    `;
}

// ============================================
// ORCHESTRATOR: Coordinates the workflow
// ============================================
async function loadUserGroups(javaURI, fetchOptions, containerElement) {
    try {
        // Step 1: Fetch and validate user
        const userData = await fetchCurrentUser(javaURI, fetchOptions);
        const { personId, uid } = validateUserIdentity(userData);
        
        // Step 2: Fetch user's groups
        const groups = await fetchUserGroups(personId, javaURI, fetchOptions);
        
        // Step 3: Filter groups (in case we got all groups)
        const userGroups = filterGroupsByMembership(groups, uid);
        
        // Step 4: Render to UI
        renderGroupsToUI(userGroups, containerElement, personId);
        
        return { personId, uid, groups: userGroups };
        
    } catch (error) {
        // Centralized error handling
        if (error.message === 'AUTHENTICATION_REQUIRED') {
            showErrorMessage(containerElement, 'Please sign in to view groups.');
        } else {
            showErrorMessage(containerElement, 'Error loading groups.');
            console.error('Group loading error:', error);
        }
        throw error;
    }
}

// ============================================
// USAGE
// ============================================
const container = document.getElementById('springGroupInfo');
loadUserGroups(javaURI, fetchOptions, container);

<IPython.core.display.Javascript object>

## Comparison: Before vs After Refactoring

### Before (Monolithic Chain)
- ❌ One massive promise chain doing everything
- ❌ Hard to test individual steps
- ❌ Error handling mixed with business logic
- ❌ Can't reuse validation or rendering logic
- ❌ Difficult to understand flow

### After (SRP-Compliant)
- ✅ 7 focused functions, each with ONE job
- ✅ Easy to test (can test `filterGroupsByMembership` in isolation)
- ✅ Reusable (use `renderGroupsToUI` for different data sources)
- ✅ Clear separation: data fetching vs validation vs rendering
- ✅ Easy to modify (change UI without touching API logic)

### Testing Benefits



In [ ]:
// Easy to test individual functions!

// Test 1: Validation
const testData1 = { id: 123, uid: "test", name: "Alice" };
console.log(validateUserIdentity(testData1)); 
// Expected: { personId: 123, uid: "test", name: "Alice" }

// Test 2: Filtering
const testGroups = [
    { id: 1, members: [{ uid: "alice" }] },
    { id: 2, members: [{ uid: "bob" }] }
];
console.log(filterGroupsByMembership(testGroups, "alice"));
// Expected: [{ id: 1, members: [...] }]

// Test 3: Edge cases
console.log(filterGroupsByMembership(null, "alice")); 
// Expected: []


<IPython.core.display.Javascript object>

## Key Takeaways

### 1. Single Responsibility Principle
- Functions should do ONE thing well
- Makes code testable, reusable, maintainable
- Easier to debug and modify

### 2. API Chaining
- Sequential API calls where each depends on previous data
- Use `.then()` chaining or `async/await`
- Always handle errors at each step
- Consider early exits to stop unnecessary work

### 3. APCSP Function Design
- **Parameters:** Customize behavior with inputs
- **Algorithm:** Clear sequenced steps
- **Return Value:** Provide usable output
- **Abstraction:** Hide complexity, expose simple interface

### 4. Practical Benefits
- **Team Development:** Clear function purposes improve collaboration
- **Testing:** Single-purpose functions are easy to unit test
- **Debugging:** Smaller functions isolate problems quickly
- **Code Reuse:** Well-designed functions work in multiple contexts

## Practice Challenges

1. **Identify SRP Violations:** Find a function in your codebase that does multiple things and refactor it

2. **Design a Chain:** Create an API chain that:
   - Fetches a product by ID
   - Uses that product's category to fetch similar products
   - Renders the results

3. **APCSP Function:** Write a function that meets all APCSP requirements:
   - Takes parameters
   - Has clear algorithm
   - Returns meaningful value
   - Demonstrates abstraction

4. **Refactor:** Take a 50+ line function and break it into 5-7 SRP-compliant functions

## Additional Resources

- **APCSP Reference:** [College Board AP CSP Framework](https://apcentral.collegeboard.org/courses/ap-computer-science-principles)
- **SRP Deep Dive:** Robert C. Martin's "Clean Code"
- **API Design:** [RESTful API Best Practices](https://restfulapi.net/)
- **Promise Chaining:** [MDN Promise Documentation](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Global_Objects/Promise)

---

**Remember:** Good code is not about being clever—it's about being clear, maintainable, and easy for others (including future you) to understand!